# ============================================
# RETRIEVER TUNING PLAYGROUND
# ============================================

This notebook helps you experiment with Top-K retrieval and Similarity Thresholds using LlamaIndex and HuggingFace Embeddings.

You'll learn how to tune retrieval parameters to get better results from your RAG system.

# ============================================

In [1]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing the Python packages needed for this notebook:
# - llama-index: The RAG framework
# - pymupdf: For reading PDF files
# - llama-index-embeddings-huggingface: For using HuggingFace embedding models
#
# WHY THIS MATTERS:
# These libraries provide the building blocks for our RAG system.
#
# WHAT YOU'LL SEE:
# Installation progress messages (the -q flag keeps output minimal)
# ============================================

!pip install -q llama-index pymupdf llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [2]:
# ============================================
# STEP 2: Import Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Loading the Python modules we'll use throughout this notebook.
#
# WHY THIS MATTERS:
# Each import provides specific functionality:
# - VectorStoreIndex: Stores document chunks as vectors for similarity search
# - Document: Wraps our text content
# - HuggingFaceEmbedding: Converts text to numerical vectors
# - SentenceSplitter: Breaks documents into smaller chunks
# - fitz (PyMuPDF): Extracts text from PDF files
#
# WHAT YOU'LL SEE:
# No output if successful (imports happen silently)
# ============================================

from llama_index.core import VectorStoreIndex, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.settings import Settings
import fitz  # PyMuPDF
import time

In [3]:
# ============================================
# STEP 3: Load the PDF Document
# ============================================
#
# WHAT WE'RE DOING:
# Loading a pharmaceutical SDF (Specification Data File) document
# and extracting all text from it.
#
# WHY THIS MATTERS:
# This is the document our RAG system will search through.
# The SDF contains product specifications, test methods, and
# quality control information - exactly what we want to query.
#
# WHAT YOU'LL SEE:
# The text variable will contain all text from the PDF.
# ============================================

# Load the pharmaceutical SDF document (upload your own or use the sample)
pdf_path = "/content/sample-sdf-document.pdf"
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the document.")

Extracted 617 words from the document.


In [4]:
# ============================================
# STEP 4: Configure the Embedding Model
# ============================================
#
# WHAT WE'RE DOING:
# Setting up the model that converts text into numerical vectors.
# We're using MiniLM-L6-v2, a fast and efficient embedding model.
#
# WHY THIS MATTERS:
# Embeddings are the foundation of semantic search. They allow
# us to find text chunks that are semantically similar to our
# query, even if they don't share exact keywords.
#
# WHAT YOU'LL SEE:
# Model download progress on first run, then silence on success.
# ============================================

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# ============================================
# STEP 5: Split Document into Chunks and Build Index
# ============================================
#
# WHAT WE'RE DOING:
# 1. Breaking the document into small, overlapping chunks
# 2. Converting each chunk into a vector embedding
# 3. Storing all vectors in a searchable index
#
# WHY THIS MATTERS:
# - Small chunks (50 tokens) allow precise retrieval
# - Overlap (50 tokens) prevents information loss at boundaries
# - The index enables fast similarity search
#
# WHAT YOU'LL SEE:
# Progress messages as chunks are processed and indexed.
# ============================================

# Define a sentence splitter (chunk_size and chunk_overlap in tokens)
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.")

# Build the vector index from these nodes
index = VectorStoreIndex(nodes)
print("Vector index built successfully!")

Created 73 chunks from the document.
Vector index built successfully!


## ============================================
## EXPERIMENT 1: Top-K Retrieval
## ============================================

Top-K controls how many chunks are returned.
- Low K (2-3): More focused, may miss relevant info
- High K (10+): More comprehensive, may include noise

## ============================================

In [6]:
# ============================================
# STEP 6: Define Query and Top-K Values to Test
# ============================================
#
# WHAT WE'RE DOING:
# Setting up a pharmaceutical query and different top_k values
# to see how many results affect retrieval quality.
#
# WHY THIS MATTERS:
# The right top_k value balances comprehensiveness vs. noise.
# Too few results may miss important info; too many dilute relevance.
#
# WHAT YOU'LL SEE:
# We'll compare results for top_k = 2, 5, and 10.
# ============================================

query = "What test methods were used for quality control?"
top_k_values = [2, 5, 10]

In [7]:
# ============================================
# STEP 7: Run Top-K Experiment
# ============================================
#
# WHAT WE'RE DOING:
# Retrieving chunks for each top_k value and displaying results.
#
# WHY THIS MATTERS:
# This shows you exactly how top_k affects what information
# your RAG system will use to generate answers.
#
# WHAT YOU'LL SEE:
# For each top_k value, you'll see the retrieved chunks.
# Notice how higher K values bring in more (but less relevant) results.
# ============================================

for top_k in top_k_values:
    print(f"\n{'='*60}")
    print(f"Results for top_k = {top_k}")
    print(f"{'='*60}")
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)
    for i, node in enumerate(nodes):
        print(f"\nResult {i+1} (Score: {node.score:.3f}):")
        print(node.get_text())
        print("-" * 50)


Results for top_k = 2

Result 1 (Score: 0.332):
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Result 2 (Score: 0.311):
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,
--------------------------------------------------

Results for top_k = 5

Result 1 (Score: 0.332):
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Result 2 (Score: 0.311):
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,
--------------------------------------------------

Result 3 (Score: 0.260):
derived ingredients or in compliance with EMA/410/01.  
Countr

## ============================================
## EXPERIMENT 2: Similarity Thresholds
## ============================================

Thresholds filter out low-quality matches.
- Higher threshold: Only very relevant chunks pass
- Lower threshold: More chunks pass, including marginal matches

**NOTE:** Scores from MiniLM typically range 0.1-0.4 for relevant content, so we use thresholds in that range.

## ============================================

In [8]:
# ============================================
# STEP 8: Retrieve Chunks for Threshold Filtering
# ============================================
#
# WHAT WE'RE DOING:
# Getting the top 10 chunks so we have a pool to filter from.
#
# WHY THIS MATTERS:
# We need to retrieve chunks first, then we can apply score
# thresholds to filter out low-relevance results.
#
# WHAT YOU'LL SEE:
# 10 chunks retrieved, each with a similarity score.
# ============================================

retriever = index.as_retriever(similarity_top_k=10)
retrieved_nodes = retriever.retrieve(query)

print(f"Retrieved {len(retrieved_nodes)} chunks. Score range: {retrieved_nodes[-1].score:.3f} - {retrieved_nodes[0].score:.3f}")

Retrieved 10 chunks. Score range: 0.227 - 0.332


In [13]:
# Print final outputs
for i, node in enumerate(retrieved_nodes):
    print(f"\nChunk {i+1} (Score: {node.score:.2f})")
    print(node.get_text())


Chunk 1 (Score: 0.33)
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.

Chunk 2 (Score: 0.31)
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,

Chunk 3 (Score: 0.26)
derived ingredients or in compliance with EMA/410/01.  
Country of Origin: USA 

Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.

Chunk 4 (Score: 0.26)
Conformance 
Bio-safety
All polymeric materials in contact with the process fluid comply with United States Pharmacopeia (USP) <88> Biological 
Reactivity Test Class VI and are free from animal derived ingredients or in compliance with

Chunk 5 (Score: 0.26)
Gradient C, Modified, ÄKTATM ready
Expiration Date:  20230126 
Product Release Criteria 
We hereby certify that the defined product has been manufactu

In [14]:
# ============================================
# STEP 9: Apply Different Similarity Thresholds
# ============================================
#
# WHAT WE'RE DOING:
# Filtering chunks based on their similarity scores.
# We test thresholds of 0.226, 0.26, and 0.30.
#
# WHAT YOU'LL SEE:
# Different numbers of chunks passing each threshold.
# Higher thresholds = fewer but more relevant results.
# ============================================

for threshold in [0.226, 0.26, 0.30]:
    filtered_nodes = [node for node in retrieved_nodes if node.score and node.score > threshold]
    print(f"\n{'='*60}")
    print(f"Threshold = {threshold}")
    print(f"{'='*60}")
    print(f"Filtered {len(filtered_nodes)} out of {len(retrieved_nodes)} total chunks.")
    for i, node in enumerate(filtered_nodes):
        print(f"\nChunk {i+1} (Score: {node.score:.3f}):")
        print(node.get_text())
        print("-" * 50)


Threshold = 0.226
Filtered 10 out of 10 total chunks.

Chunk 1 (Score: 0.332):
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Chunk 2 (Score: 0.311):
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,
--------------------------------------------------

Chunk 3 (Score: 0.260):
derived ingredients or in compliance with EMA/410/01.  
Country of Origin: USA 

Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Chunk 4 (Score: 0.259):
Conformance 
Bio-safety
All polymeric materials in contact with the process fluid comply with United States Pharmacopeia (USP) <88> Biological 
Reactivity Test Class VI and are free from animal

## ============================================
## EXPERIMENT 3: Combined Configurations
## ============================================

Now let's try different combinations of top_k AND threshold to find the optimal retrieval configuration.

## ============================================

In [15]:
# ============================================
# STEP 10: Define Experiment Configurations
# ============================================
#
# WHAT WE'RE DOING:
# Setting up three different retrieval configurations to compare.
#
# WHY THIS MATTERS:
# Real-world RAG systems need tuned parameters. These experiments
# help you understand the tradeoffs.
#
# Configurations:
# 1. top_k=5, no threshold - Get 5 best chunks regardless of score
# 2. top_k=8, threshold=0.15 - Get up to 8, but filter weak matches
# 3. top_k=5, threshold=0.20 - Conservative: few chunks, high quality
# ============================================

experiments = [
    {"top_k": 5, "threshold": None},
    {"top_k": 8, "threshold": 0.26},
    {"top_k": 5, "threshold": 0.29},
]

In [16]:
# ============================================
# STEP 11: Run Combined Experiments
# ============================================
#
# WHAT WE'RE DOING:
# Running each configuration and comparing the results.
#
# WHY THIS MATTERS:
# This is how you would tune a production RAG system -
# testing different configurations to find what works best
# for your specific documents and query types.
#
# WHAT YOU'LL SEE:
# Results for each configuration showing chunks retrieved
# and their similarity scores.
# ============================================

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Experiment: top_k={exp['top_k']}, threshold={exp['threshold']}")
    print(f"{'='*60}")

    retriever = index.as_retriever(similarity_top_k=exp["top_k"])
    nodes = retriever.retrieve(query)

    if exp["threshold"]:
        nodes = [node for node in nodes if node.score and node.score > exp["threshold"]]

    print(f"Chunks Retrieved: {len(nodes)}")
    for i, node in enumerate(nodes):
        print(f"\nChunk {i+1} (Score: {node.score:.3f}):")
        print(node.get_text())
        print("-" * 50)


Experiment: top_k=5, threshold=None
Chunks Retrieved: 5

Chunk 1 (Score: 0.332):
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Chunk 2 (Score: 0.311):
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,
--------------------------------------------------

Chunk 3 (Score: 0.260):
derived ingredients or in compliance with EMA/410/01.  
Country of Origin: USA 

Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Chunk 4 (Score: 0.259):
Conformance 
Bio-safety
All polymeric materials in contact with the process fluid comply with United States Pharmacopeia (USP) <88> Biological 
Reactivity Test Class VI and are free from anim

## ============================================
## NEXT STEPS
## ============================================

Now you can experiment on your own:
1. Try different queries about the SDF document
2. Adjust top_k values (try 3, 7, 15)
3. Experiment with threshold values (0.12, 0.18, 0.25)
4. Try different embedding models (see the comparison notebook)

**Key takeaways:**
- Higher top_k = more context but potentially more noise
- Higher threshold = stricter filtering but may miss relevant info
- The best configuration depends on your specific use case

## ============================================